# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 02: Dataset Cleaning, Normalization & Group-Aware Splitting

**Goal**: Execute the complete cleaning pipeline on `Com-VID_data.csv`, produce `ComVID_cleaned.csv`, enforce group-aware train/val/test splits (strictly preventing video-level data leakage), and generate the formal quality audit report `dataset_audit.json`.

**Rules Enforced**:
1. Preserve original columns and values.
2. Normalize column names.
3. Clean missing values and whitespace.
4. Normalize known spelling variations in aspect and emotion labels without merging distinct concepts.
5. Flag duplicate IDs and video URLs without deleting records.
6. Guarantee 0% video leakage between train, val, and test splits.
7. Produce `reports/dataset_audit.json`.

In [ ]:
# Environment & Module Path Setup
import os
import sys
import json
from pathlib import Path

# Ensure project root is in sys.path
CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.append(str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.append(str(CURRENT_DIR.parent))

from src.preprocessing.normalizers import (
    parse_html_rating, clean_text_whitespace, normalize_emotion, normalize_aspects
)
from src.preprocessing.dataset_cleaner import ComVIDCleaner
from src.preprocessing.audit import DatasetAuditor

print('Preprocessing modules imported successfully!')

### 1. Execute ComVIDCleaner Pipeline
We run `ComVIDCleaner`, which:
- Preserves all raw fields (`raw_id`, `raw_title`, `raw_rating`, etc.).
- Extracts clean float ratings from HTML.
- Normalizes whitespace and text formatting.
- Normalizes emotion and aspect labels.
- Assigns group-aware splits based on `video_url_hash`.

In [ ]:
# Dynamic input/output path resolution
input_path = Path('data/Com-VID_data.csv')
if not input_path.exists() and Path('/kaggle/input/comvid-dataset/Com-VID_data.csv').exists():
    input_path = Path('/kaggle/input/comvid-dataset/Com-VID_data.csv')

output_path = Path('data/processed/ComVID_cleaned.csv')
report_path = Path('reports/dataset_audit.json')

# Instantiate and execute cleaner
cleaner = ComVIDCleaner(
    raw_csv_path=input_path,
    output_csv_path=output_path,
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    random_seed=42
)
cleaned_records = cleaner.process()
saved_csv = cleaner.save_cleaned_csv()

print(f'Successfully processed and saved {len(cleaned_records)} records to {saved_csv}')

### 2. Verify Preserved Raw Columns & Normalized Fields
Let's inspect the first record to verify that both raw columns and normalized columns coexist.

In [ ]:
import csv

with open(output_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    sample = next(reader)

print('Normalized Fields:')
norm_keys = ['review_id', 'title', 'rating_stars', 'emotion', 'aspects', 'domain', 'split', 'is_duplicate_id', 'is_duplicate_url']
for k in norm_keys:
    print(f'  {k:20s}: {sample.get(k)}')

print('\nPreserved Raw Fields:')
raw_keys = ['raw_id', 'raw_rating', 'raw_emotion', 'raw_aspect']
for k in raw_keys:
    print(f'  {k:20s}: {repr(sample.get(k))}')

### 3. Verification of Zero Video Leakage Across Splits
A critical validation check: verify that no video URL or video hash appears in more than one split.

In [ ]:
from collections import defaultdict

split_by_hash = {}
leakage_detected = False

with open(output_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        h = row['video_url_hash']
        s = row['split']
        if h in split_by_hash and split_by_hash[h] != s:
            print(f'LEAKAGE DETECTED! Video {h} present in both {split_by_hash[h]} and {s}')
            leakage_detected = True
        split_by_hash[h] = s

if not leakage_detected:
    print('VERIFICATION PASSED: 0% Video Leakage! All videos cleanly isolated by split.')

### 4. Generate Formal Dataset Audit Report (`reports/dataset_audit.json`)
Now run `DatasetAuditor` to produce the full audit report.

In [ ]:
auditor = DatasetAuditor.from_csv(output_path)
report = auditor.generate_report()
auditor.save_report(report_path)

print(f'Audit report generated at: {report_path.resolve()}')
print('\nDataset Audit Executive Summary:')
print(f"  Total records: {report['metadata']['total_records']}")
print(f"  Unique Review IDs: {report['identifiers_and_duplicates']['unique_review_ids']}")
print(f"  Unique Video URLs: {report['identifiers_and_duplicates']['unique_video_urls']}")
print(f"  Duplicate Video URLs: {report['identifiers_and_duplicates']['duplicate_video_urls_count']}")
print(f"  Zero-Leakage Status: {report['splits']['video_leakage_check']['is_leakage_free']}")
print('\nSplit Counts and Percentages:')
for split, d in report['splits']['records_per_split'].items():
    print(f"  - {split:5s}: {d['count']} records ({d['percentage']}%) across {report['splits']['unique_video_groups_per_split'][split]} video groups")

### Conclusion of Phase 1
Phase 1 data cleaning, normalization, duplicate auditing, and leak-free group-aware splitting are complete and verified. Outputs:
- `data/processed/ComVID_cleaned.csv`
- `reports/dataset_audit.json`

Ready for Phase 2: Video Downloading and Validation.